# Assignment 32 : AI Agents using Langchain

# PART 1 Tools in AI Agents

# TAsk 1 : Understanding Tools
## Answer briefly
## 1. What is a tool in an AI Agent

### A tool is a function or external service that an AI agent uses to perform specific tasks, such as searching the web, running code, or accessing a database.

## 2. Why do agents need tools?

### Agents need tools to access real-time information, perform calculations, interact with external systems, and complete tasks beyond their built-in knowledge.

## 3. Difference between a chatbot and an agent.


## Chatbot                        |	AI Agent
### Mainly responds to user messages. |	Takes actions to achieve goals.
### Usually provides text-based answers.|	Can use tools and external services.
### Mostly follows a question-and-answer approach.|	Can plan, make decisions, and execute multiple steps.

# Task 2: Built in tools in langchain
## Use at least 3 built in tools , such as:
## Calculator
## Wikipedia
## Web search
## 1. Initialize tools
## 2. Test each tool individually
## 3. Print tool Outputs.

In [2]:
!pip install -U langchain langchain-community wikipedia ddgs numexp

ERROR: Could not find a version that satisfies the requirement numexp (from versions: none)
ERROR: No matching distribution found for numexp


In [6]:


from langchain_community.tools import DuckDuckGoSearchRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_core.tools import tool
import numexpr


# 1. Initialize tools

# Calculator
@tool
def calculator(expression: str) -> str:
    """Evaluate a math expression, e.g. '37593 * 67' or 'sqrt(144) + 2**5'."""
    try:
        return str(numexpr.evaluate(expression.strip()).item())
    except Exception as e:
        return f"Error: {e}"


# Wikipedia
wikipedia = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=500)
)

# Web search
search = DuckDuckGoSearchRun()

tools = [calculator, wikipedia, search]


In [8]:

# 2. Test each tool individually

# Tool 1: Calculator
calc_input = "37593 * 67 + sqrt(144)"
calc_output = calculator.invoke(calc_input)

# Tool 2: Wikipedia
wiki_input = "Search Wikipedia for general information about people, places, concepts and other topics."
wiki_output = wikipedia.invoke(wiki_input)

# Tool 3: Web search
search_input = "latest developments in generative AI"
search_output = search.invoke(search_input)


In [11]:
## Print outputs
print(f"The calculator tool output is :{calc_output}")
print(f"The wikipedia tool output is :{wiki_output}")
print(f"The web search tool output is :{search_output}")


The calculator tool output is :2518743.0
The wikipedia tool output is :Page: Health information on Wikipedia
Summary: The Wikipedia online encyclopedia has, since the late 2000s, served as a popular source for health information for both laypersons and, in many cases, health care practitioners. Health-related articles on Wikipedia are popularly accessed as results from search engines, which frequently deliver links to Wikipedia articles. Independent assessments have been made of the number and demographics of people who seek health information on Wikipedia, the sco
The web search tool output is :Explore the latest news and expert commentary on Generative AI, brought to you by the editors of AI Business From reshaping industries to empowering individuals with creative tools, Generative AI continuously pushes the boundaries of what machines can achieve. This article serves as your one-stop destination for staying updated with the latest developments, breakthroughs, and trends in the ever

# PART 2 -- Creating Custom Tools & Toolkits

# Task 3: Create a Custom tool
## @tool
## def company_policy_lookup(query:str)-> str:
## """Returns company policy information"""
## 1. Implement logic (mock data is fine)
## 2. Test the tool independently.

In [13]:
from langchain.tools import tool

@tool
def company_policy_lookup(query: str) -> str:
    """Returns company policy information based on the user's query."""

    policies = {
        "leave": "Employees receive  paid annual leave according to company rules.",
        "work from home": "Employees can work from home with manager approval.",
        "working hours": "Regular working hours are 9 AM to 5 PM, Monday to Friday.",
        "dress code": "Employees should follow the company's professional dress code."
    }

    query = query.lower()

    for key, value in policies.items():
        if key in query:
            return value

    return "Sorry, no matching company policy was found."


# Test the tool independently
result = company_policy_lookup.invoke({"query": "What is the work from home policy?"})

# Print tool output
print("Tool Name:", company_policy_lookup.name)
print("Tool Description:", company_policy_lookup.description)
print("Tool Output:", result)

Tool Name: company_policy_lookup
Tool Description: Returns company policy information based on the user's query.
Tool Output: Employees can work from home with manager approval.


# Task 4: Create a Custom Toolkit
## 1. Group multiple tools into a toolkit.
## 2. Example tools:
## . Policy lookup
## . Simple database query
## . Date/time tool

In [16]:
from langchain.tools import tool
from datetime import datetime

# database
employees = {
    "ani": {"department": "IT", "salary": 25000},
    "rahul": {"department": "HR", "salary": 30000},
    "amit": {"department": "Finance", "salary": 28000}
}


# Tool 1: Policy Lookup
@tool
def company_policy_lookup(query: str) -> str:
    """Returns company policy information based on the query."""

    policies = {
        "leave": "Employees can apply for leave with manager approval.",
        "work from home": "Work from home requires manager approval.",
        "working hours": "Working hours are 9 AM to 5 PM."
    }

    query = query.lower()

    for key, value in policies.items():
        if key in query:
            return value

    return "No matching company policy found."


# Tool 2: Simple Database Query
@tool
def employee_database_query(name: str) -> str:
    """Returns  employee information using an employee name."""

    name = name.lower().strip()

    if name in employees:
        data = employees[name]
        return name.title(),data['department'],data['salary']
    else:
        return "Employee not found."


# Tool 3: Date and Time
@tool
def current_datetime() -> str:
    """Returns the current date and time."""

    return datetime.now().astimezone().strftime(
        "%d-%m-%Y %I:%M:%S %p %Z"
    )


# Create a custom toolkit by grouping tools
company_toolkit = [
    company_policy_lookup,
    employee_database_query,
    current_datetime
]

# PART 3 --- Tool Binding & Tool Calling

# Task 5 : Tool Binding to LLM
## 1. Bind tools to an LLM using Langchain.
## 2. Allow the LLM to decide when to call which tool.


In [3]:
from langchain_groq import ChatGroq
from langchain.tools import tool
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()

# Initialize LLM
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

# Tool 1: Company Policy
@tool
def company_policy_lookup(query: str) -> str:
    """Returns company policy information based on the query."""

    policies = {
        "leave": "Employees can apply for leave with manager approval.",
        "work from home": "Work from home requires manager approval.",
        "working hours": "Working hours are 9 AM to 5 PM."
    }

    query = query.lower()

    for key, value in policies.items():
        if key in query:
            return value

    return "No matching company policy found."


# Tool 2: Employee Database
employees = {
    "ani": {"department": "IT", "salary": 25000},
    "rahul": {"department": "HR", "salary": 30000}
}

@tool
def employee_database_query(name: str) -> str:
    """Returns mock employee information using employee name."""

    name = name.lower().strip()

    if name in employees:
        data = employees[name]
        return (
            f"Department: {data['department']}, "
            f"Salary: Rs. {data['salary']}"
        )

    return "Employee not found."


# Tool 3: Current Date and Time
@tool
def current_datetime() -> str:
    """Returns the current date and time."""

    return datetime.now().astimezone().strftime(
        "%d-%m-%Y %I:%M:%S %p %Z"
    )


# Bind tools to the LLM
tools = [
    company_policy_lookup,
    employee_database_query,
    current_datetime
]

llm_with_tools = llm.bind_tools(tools)

# Ask the LLM a question
response = llm_with_tools.invoke(
    "What is the work from home policy?"
)

# Print response
print("LLM Response:",response.content)
print("\nTool Calls:",response.tool_calls)

LLM Response: 

Tool Calls: [{'name': 'company_policy_lookup', 'args': {'query': 'work from home policy'}, 'id': 'fc_1f6fd46e-ccf9-4b4b-9070-2d4f313e3527', 'type': 'tool_call'}]


# Task 6 --Tool Calling Flow
## Demonstrate:
## User Query ---> LLM ---> Tool Selection ---> Tool Execution ----> Final Answer
## Test with queries that require:
## Single tool
## Multiple tools

# 1. User Query ---> 
## "What is Ani's department and the leave policy?"

# 2. LLM --->
## Understands the question and decides which tools are needed.

# 3. Tool Selection --->
## Selects employee database + policy lookup.

# 4. Tool Execution --->
## Executes both tools and collects their results.

# 5. Final Answer --->
## LLM combines the tool results into a response.

In [ ]:
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain_core.messages import ToolMessage
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()

# Initialize LLM
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

# Mock employee database
employees = {
    "ani": {"department": "IT", "salary": 25000},
    "rahul": {"department": "HR", "salary": 30000}
}

# Tool 1: Policy Lookup
@tool
def company_policy_lookup(query: str) -> str:
    """Look up company policies about leave, work from home, and working hours."""

    policies = {
        "leave": "Employees can apply for leave with manager approval.",
        "work from home": "Work from home requires manager approval.",
        "working hours": "Working hours are 9 AM to 5 PM."
    }

    query = query.lower()

    for key, value in policies.items():
        if key in query:
            return value

    return "No matching company policy found."


# Tool 2: Employee Database
@tool
def employee_database_query(name: str) -> str:
    """Look up an employee's department and salary by name."""

    name = name.lower().strip()

    if name in employees:
        data = employees[name]
        return (
            f"Name: {name.title()}, "
            f"Department: {data['department']}, "
            f"Salary: Rs. {data['salary']}"
        )

    return "Employee not found."


# Tool 3: Date and Time
@tool
def current_datetime() -> str:
    """Return the current date and time."""

    return datetime.now().astimezone().strftime(
        "%d-%m-%Y %I:%M:%S %p %Z"
    )


tools = [
    company_policy_lookup,
    employee_database_query,
    current_datetime
]

tool_map = {t.name: t for t in tools}
llm_with_tools = llm.bind_tools(tools)


def run_query(query: str):
    print("\nUSER QUERY:", query)
    print("\n--- TOOL CALLING FLOW ---")

    messages = [("user", query)]

    # Allow multiple rounds of tool calls
    for step in range(5):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            print("\nFINAL ANSWER:")
            print(response.content)
            return

        # Execute each selected tool
        for call in response.tool_calls:
            name = call["name"]
            args = call["args"]

            print(f"\nTOOL SELECTED: {name}")
            print("TOOL INPUT:", args)

            result = tool_map[name].invoke(args)

            print("TOOL OUTPUT:", result)

            messages.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=call["id"]
                )
            )


# Test 1: Single tool
run_query("What is the work from home policy?")

# Test 2: Multiple tools
run_query(
    "What is Ani's department and salary, "
    "and what is the company leave policy?"
)

# Test 3: Multiple tools 
run_query(
    "Tell me Rahul's department and the current date and time."
)


USER QUERY: What is the work from home policy?

--- TOOL CALLING FLOW ---

TOOL SELECTED: company_policy_lookup
TOOL INPUT: {'query': 'work from home'}
TOOL OUTPUT: Work from home requires manager approval.

FINAL ANSWER:
The company’s work‑from‑home (WFH) policy is straightforward:

**Employees may work from home only with prior approval from their manager.**  

If you’re interested in arranging a WFH day (or a regular remote schedule), you should discuss it with your manager and obtain their sign‑off before making any arrangements.

USER QUERY: What is Ani's department and salary, and what is the company leave policy?

--- TOOL CALLING FLOW ---

TOOL SELECTED: employee_database_query
TOOL INPUT: {'name': 'Ani'}
TOOL OUTPUT: Name: Ani, Department: IT, Salary: Rs. 25000

TOOL SELECTED: company_policy_lookup
TOOL INPUT: {'query': 'leave policy'}
TOOL OUTPUT: Employees can apply for leave with manager approval.

FINAL ANSWER:
**Ani’s Details**  
- **Department:** IT  
- **Salary:** Rs. 

# PART 4 -- Creating  a React AI Agent


# Task 7 : React Agent Overview
## Explain
## 1. What is ReAct(Reason + Act)?
## 2. Why React agents are powerful.

## 1. What is ReAct (Reason + Act)?
### ReAct stands for Reasoning and Acting. It is an approach in AI agents where an LLM alternates between reasoning about a task, taking actions using tools, and observing the results until it can provide a final answer.

## 2. Why are ReAct agents powerful?
###  They can use APIs, databases, calculators, and web search like tools.
###  They can choose the next action based on the information available.
### .They can break tasks into multiple steps and handles the task .
### .They can use tool results to guide sequent actions and makes Feedback-based improvement.
### .They can retrieve external information instead of relying only on the LLM's existing knowledge.

# Task 8 : Build a ReAct Agent
## 1. Create a ReAct-style prompt.
## 2. Attach tools to the agent.
## 3. Allow  the agent to :
## . Think
## . Act(call tool)
## . Observe
## . Answer

In [8]:
from datetime import datetime
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain_classic.agents import AgentExecutor, create_react_agent

load_dotenv()

# Initialize LLM
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

# employee database
employees = {
    "ani": {"department": "IT", "salary": 25000},
    "rahul": {"department": "HR", "salary": 30000}
}


# Tool 1: Company Policy Lookup
@tool
def company_policy_lookup(query: str) -> str:
    """Look up company policies about leave, work from home,
    and working hours."""

    policies = {
        "leave": "Leave requires manager approval.",
        "work from home": "Work from home requires manager approval.",
        "working hours": "Working hours are 9 AM to 5 PM."
    }

    query = query.lower()

    for key, value in policies.items():
        if key in query:
            return value

    return "No matching policy found."


# Tool 2: Employee Database
@tool
def employee_database_query(name: str) -> str:
    """Find an employee's department and salary by name."""

    name = name.lower().strip()

    if name in employees:
        data = employees[name]
        return (
            f"Department: {data['department']}, "
            f"Salary: Rs. {data['salary']}"
        )

    return "Employee not found."


# Tool 3: Date and Time
@tool
def current_datetime() -> str:
    """Return the current local date and time."""

    return datetime.now().astimezone().strftime(
        "%d-%m-%Y %I:%M:%S %p %Z"
    )


# Attach tools to the agent
tools = [
    company_policy_lookup,
    employee_database_query,
    current_datetime
]


#  ReAct-style prompt
template = """
You are a helpful company assistant.

Answer the user's question using the available tools when needed.
Use tool results as evidence. Do not invent missing information.

You can use the following tools:
{tools}

Follow this format:

Question: the user's question
Thought: briefly identify what information or action is needed
Action: the tool name, exactly as listed below
Action Input: the input for the tool
Observation: the tool's returned result
... repeat if another tool is needed ...
Thought: determine whether enough information is available
Final Answer: the answer to the user

Available tool names: {tool_names}

Question: {input}
Thought: {agent_scratchpad}
"""

prompt = PromptTemplate.from_template(template)


# Create ReAct agent
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)


# Execute the agent
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=5,
    handle_parsing_errors=True
)


# Test 1: Single tool
print("\nSINGLE TOOL")

result1 = agent_executor.invoke({
    "input": "What is the work from home policy?"
})

print("\nFinal Answer:", result1["output"])


# Test 2: Multiple tools
print("\nTEST 2: MULTIPLE TOOLS ")

result2 = agent_executor.invoke({
    "input": (
        "Find Ani's department and salary, "
        "and tell me the company leave policy."
    )
})

print("\nFinal Answer:", result2["output"])


# Test 3: Date/time tool
print("\n DATE AND TIME ")

result3 = agent_executor.invoke({
    "input": "What is the current date and time?"
})

print("\nFinal Answer:", result3["output"])

c:\Users\Anish Rawat\GEN_AI\Langchain Projects\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



SINGLE TOOL


> Entering new AgentExecutor chain...
Question: What is the work from home policy?
Thought: I need to retrieve the company's work from home policy using the policy lookup tool.
Action: company_policy_lookup
Action Input: work from homeWork from home requires manager approval.**Answer:**  
Our company's work‑from‑home policy states that any work‑from‑home arrangement must be approved by the employee’s manager. If you’re interested in working from home, please discuss it with your manager to obtain the necessary approval and to learn about any additional guidelines that may apply.Invalid Format: Missing 'Action:' after 'Thought:'Invalid Format: Missing 'Action:' after 'Thought:'Question: What is the work from home policy?

Thought: I need to retrieve the company's work‑from‑home policy using the policy lookup tool.

Action: company_policy_lookup
Action Input: work from home policy details
Work from home requires manager approval.Question: What is the work from home policy?

# Task 9: Testing the ReAct Agent
## Test the agent with:
## 1. Factual question
## 2. Calculation based question
## 3. Multi-step reasoning question
## Observe the reasoning trace.

In [10]:
# TEST 1: Factual Question
print("\n TEST 1: FACTUAL QUESTION ")

result1 = agent_executor.invoke({
    "input": "What is the company leave policy?"
})

print("\nFinal Answer:", result1["output"])


# TEST 2: Calculation-Based Question
print("\nTEST 2: CALCULATION")

result2 = agent_executor.invoke({
    "input": "Calculate (25000 + 30000) * 12 using the calculator tool."
})

print("\nFinal Answer:", result2["output"])


# TEST 3: Multi-Step Reasoning
print("\nMULTI-STEP REASONING")

result3 = agent_executor.invoke({
    "input": (
        "Find Ani's monthly salary from the employee database, "
        "calculate her annual salary, and tell me the leave policy."
    )
})

print("\nFinal Answer:", result3["output"])


 TEST 1: FACTUAL QUESTION 


> Entering new AgentExecutor chain...
Question: What is the company leave policy?
Thought: I need to retrieve the company's leave policy from the policy database.
Action: company_policy_lookup
Action Input: "leave policy"Leave requires manager approval.Question: What is the company leave policy?
Thought: The initial lookup returned only a brief note that leave requires manager approval. I need more comprehensive details about the leave policy, such as types of leave, accrual rates, eligibility, and any other relevant rules.
Action: company_policy_lookup
Action Input: "comprehensive leave policy"Leave requires manager approval.Question: What is the company leave policy?  
Thought: The policy lookup returned only a brief statement that “Leave requires manager approval.” No additional details are available in the system. I should provide the information I have and advise the user to consult HR or the employee handbook for comprehensive details.  
Final Answer

# PART 5 --Mini Project: AI Agent Assistent


# Task 10 : Agent Use Case
## Build an agent that can 
## Answer questions
## Perform calculations
## Retrieve information
## Agent should automatically choose tools.

In [ ]:

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_core.prompts import PromptTemplate

import ast
import operator

load_dotenv()

# Initialize LLM
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

# Mock employee database
employees = {
    "ani": {"department": "IT", "salary": 25000},
    "rahul": {"department": "HR", "salary": 30000},
    "amit": {"department": "Finance", "salary": 28000}
}


# Tool 1: Answer company policy questions
@tool
def policy_lookup(query: str) -> str:
    """Retrieve company policy information about leave,
    work from home, and working hours."""

    policies = {
        "leave": "Leave requires manager approval.",
        "work from home": "Work from home requires manager approval.",
        "working hours": "Working hours are 9 AM to 5 PM."
    }

    query = query.lower()

    for key, value in policies.items():
        if key in query:
            return value

    return "No matching company policy found."


# Tool 2: Retrieve employee information
@tool
def employee_lookup(name: str) -> str:
    """Retrieve an employee's department and salary by name."""

    name = name.lower().strip()

    if name not in employees:
        return "Employee not found."

    data = employees[name]

    return (
        f"Name: {name.title()}, "
        f"Department: {data['department']}, "
        f"Monthly salary: Rs. {data['salary']}"
    )


# Tool 3: Perform calculations safely


# Calculator Tool

@tool
def calculator(expression: str) -> str:
    """
    Calculate a mathematical expression.
    Example: 25 * 40
    """

    try:
        # Remove unnecessary spaces
        expression = expression.strip()

        result = eval(
            expression,
            {"__builtins__": {}},
            {}
        )

        return str(result)

    except Exception as e:
        return f"Error calculating expression: {e}"

tools = [policy_lookup, employee_lookup, calculator]

# ReAct prompt
template = """
You are a helpful company assistant.

Choose the appropriate tool automatically based on the user's question.
Use retrieved tool results as evidence. Never invent employee records
or company policies. Use the calculator for arithmetic.

Available tools:
{tools}

Use this format:
Question: the user's question
Thought: briefly identify what is needed
Action: one tool name from [{tool_names}]
Action Input: the input for that tool
Observation: the tool result
... repeat if needed ...
Final Answer: a clear answer based on the results

Question: {input}
Thought: {agent_scratchpad}
"""

prompt = PromptTemplate.from_template(template)

# Create agent
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=6,
    handle_parsing_errors=True
)


#question-answering loop
print("AI Agent Ready! Type 'exit' to stop.")

while True:
    query = input("\nYou: ").strip()

    if query.lower() in ["exit", "quit"]:
        print("Agent stopped.")
        break

    if not query:
        continue

    try:
        result = agent_executor.invoke({"input": query})
        print("\nAgent:", result["output"])

    except Exception as e:
        print("Error:", str(e))


AI Agent Ready! Type 'exit' to stop.


> Entering new AgentExecutor chain...
**Question:** tell me something about ai  
**Thought:** This request is for general information about artificial intelligence, which doesn’t require any of the available company‑specific tools. I can answer directly.  

**Final Answer:**  
Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks typically requiring human intelligence. These tasks include learning from data (machine learning), recognizing patterns (computer vision), understanding and generating natural language (natural language processing), making decisions, and solving problems. AI can be categorized in several ways:

1. **Narrow (or Weak) AI:** Designed for a specific task, such as voice assistants (e.g., Siri, Alexa), recommendation engines, or image‑recognition software.  
2. **General (or Strong) AI:** A theoretical form of AI that would possess the ability to understand, learn, and apply knowledge

# Task 11 : Observations & Insights
## Write short answers
## 1. Benefits of tool-augmented agents
## 2. Challenges with agents
## 3. Difference between chains and agents
## 4. When to use agents over RAG

## 1. Benefits of Tool-Augmented Agents
### .They can perform calculations, search data, call APIs, and execute tasks.
### .They improve accuracy and automate complex workflows.

## 2. Challenges with Agents
### They may produce incorrect results or choose the wrong tools.
### They can be complex to debug, slow, and costly.

## 3. Difference Between Chains and Agents
### .Chains: Follow a predefined sequence of steps.
### .Agents: Dynamically decide which steps and tools to use based on the task.

## 4. When to Use Agents Over RAG
### . Use agents when tasks require multiple steps, decision-making, or tool execution.
### .Use RAG when answers mainly require retrieving information from documents or a knowledge base.